# A. Electricity, battery storage and hydrogen

Build five dispatch cases by adding electricity, a battery, constant hydrogen export, a hydrogen tank and an explicit water input. Watch how the stores use the same renewable electricity.

Run the cells from top to bottom. The examples use one node and 24 hours of synthetic data.
Source links refer to [PyPSA-Earth](https://github.com/pypsa-meets-earth/pypsa-earth/tree/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f).

## Set up the notebook

In a fresh Colab CPU session, uncomment and run the installation cell before importing packages.
If you have already imported numerical packages, restart the runtime after installation.
A local environment with `requirements-colab.txt` can skip this cell.

In [3]:
pip install -q pypsa==1.2.4 linopy==0.8.0 highspy==1.15.1 "numpy<2.5" pandas==2.2.3 "plotly>=5.24.1,<6" PyYAML==6.0.3

In [4]:
from pathlib import Path

import pypsa
import pandas as pd
import plotly.express as px
import yaml

pd.options.plotting.backend = "plotly"
px.defaults.template = "plotly_white"

SOLVER = "highs"
pypsa.options.params.optimize.include_objective_constant = True
pypsa.options.params.statistics.nice_names = False

The next cell uses local tutorial data or downloads it automatically from GitHub.

In [5]:
data_dir = Path("tutorial-data")
if not data_dir.exists() and Path("../tutorial-data").exists():
    data_dir = Path("../tutorial-data")
if not data_dir.exists():
    import zipfile
    from urllib.request import urlretrieve

    course_ref = "1f25795897e14d1307003110a6b6587f21155278"
    url = f"https://raw.githubusercontent.com/ljansen-iee/pypsa-meets-earth-lab-2026/{course_ref}/tutorial-data.zip"
    archive_path = Path("tutorial-data.zip")
    urlretrieve(url, archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(".")

config = yaml.safe_load((data_dir / "config/config.DEMO_BASE.yaml").read_text())

data_dir = data_dir.resolve()
work_dir = data_dir.parent / "work"

## 1. Read the electricity inputs

The electricity bus is `ZA0`; the name represents one teaching cluster.
Generators use the workflow's `<node> <carrier>` pattern. The AC load here is an
illustrative non-industrial profile; industry electricity is added separately below.

Source: [renewable generator names and attributes](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/add_electricity.py#L710-L734);
[demand-profile CSV output](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/Snakefile#L582-L610).
The renewable CSV replaces the workflow's weather-derived NetCDF profiles for this lesson.

In [6]:
load = pd.read_csv(data_dir / "resources/demand_profiles.csv", index_col=0, parse_dates=True)
load.head()  # MW_el at each hourly snapshot.

,ZA0
snapshot,
2013-01-01 00:00:00,3.000000
2013-01-01 01:00:00,3.000003
2013-01-01 02:00:00,3.000074
2013-01-01 03:00:00,3.001158
2013-01-01 04:00:00,3.010989


In [7]:
profiles = pd.read_csv(data_dir / "resources/renewable_profiles/profiles_24h.csv", index_col=0, parse_dates=True)
# Capacity factors are dimensionless fractions multiplying p_nom.
profiles

,solar,onwind
snapshot,,
2013-01-01 00:00:00,0.000000e+00,0.35
2013-01-01 01:00:00,0.000000e+00,0.35
2013-01-01 02:00:00,0.000000e+00,0.35
2013-01-01 03:00:00,0.000000e+00,0.35
2013-01-01 04:00:00,0.000000e+00,0.35
2013-01-01 05:00:00,0.000000e+00,0.35
2013-01-01 06:00:00,0.000000e+00,0.22
2013-01-01 07:00:00,2.588190e-01,0.22
2013-01-01 08:00:00,5.000000e-01,0.22


In [8]:
elec_costs = pd.read_csv(data_dir / "resources/costs_2035_elec.csv", index_col=0)
costs = pd.read_csv(data_dir / "resources/costs_2035_sec.csv", index_col=0)
costs


,annual_fixed,efficiency,fixed,capital_cost,marginal_cost,lifetime
technology,,,,,,
solar,50000,1.0,136.986301,136.986301,0.0,25
onwind,110000,1.0,301.369863,301.369863,0.0,25
OCGT,40000,0.4,109.589041,109.589041,115.5,25
battery,97000,1.0,265.753425,265.753425,0.1,25
battery inverter,25000,0.9,68.493151,68.493151,0.0,25
electrolysis,60000,0.7,164.383562,164.383562,0.0,25
fuel cell,90000,0.5,246.575342,246.575342,0.1,25
hydrogen storage tank type 1 including compressor,1000,1.0,2.739726,2.739726,0.0,25


In [9]:
nodes = pd.Index(["ZA0"], name="Bus")
snapshots = load.index
n = pypsa.Network()
n.set_snapshots(snapshots)

### Carrier colours for plotting

Carrier entries assign plotting colours. Buses and conversion Links define the physical balances.

In [10]:
n.add(
    "Carrier",
    ["AC", "solar", "onwind", "OCGT", "industry electricity", "battery",
     "H2", "H2 Electrolysis", "H2 export", "H2 Store Tank", "water", "water supply"],
    color=["#a6bbc8", "#fdb913", "#005b7f", "#d3c7ae", "#a6bbc8", "#836bad",
           "#179c7d", "#179c7d", "#4CC2A6", "#179c7d", "#39c1cd", "#39c1cd"],
)

In [11]:
n.add("Bus", nodes, carrier="AC", location=nodes, country="ZA", x=25.0, y=-29.0)
n.add("Load", nodes, bus=nodes, carrier="AC", p_set=load)

for carrier, capacity in {"solar": 12.0, "onwind": 8.0}.items():
    n.add("Generator", nodes,
          suffix=" " + carrier,
          bus=nodes,
          carrier=carrier,
          p_nom=capacity,
          p_max_pu=profiles[carrier].to_frame(nodes[0]),
          capital_cost=elec_costs.at[carrier, "capital_cost"],
          marginal_cost=elec_costs.at[carrier, "marginal_cost"],
          efficiency=elec_costs.at[carrier, "efficiency"])

# Keep thermal supply as one Generator to keep the electricity system small.
# The sector workflow converts conventional generators to fuel-input Links.
n.add("Generator", nodes + " OCGT",
      bus=nodes,
      carrier="OCGT",
      p_nom=12.0,
      efficiency=elec_costs.at["OCGT", "efficiency"],
      marginal_cost=elec_costs.at["OCGT", "marginal_cost"],
      capital_cost=elec_costs.at["OCGT", "capital_cost"])
n.generators[["bus", "carrier", "p_nom"]]

,bus,carrier,p_nom
name,,,
ZA0 solar,ZA0,solar,12.0
ZA0 onwind,ZA0,onwind,8.0
ZA0 OCGT,ZA0,OCGT,12.0


### Add the standard industrial electricity Load

[add_industry reads generated demand](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/prepare_sector_network.py#L1812-L1815)
and [adds industry electricity at annual MWh / 8760](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/prepare_sector_network.py#L2027-L2036).
The CSV in this pack is a synthetic output of the standard CAGR calculation explored in B.
The example's hydrogen column is zero.

In [12]:
# Annual MWh / 8,760 h gives a constant MW Load, even for this 24-hour subset.
industrial_demand = pd.read_csv(
    data_dir / "resources/demand/industrial_energy_demand_per_node_elec_s_1_2035.csv", index_col=0)

n.add("Load", nodes, suffix=" industry electricity", bus=nodes, carrier="industry electricity",
      p_set=industrial_demand.loc[nodes, "electricity"] / 8760)

n.loads[["bus", "carrier", "p_set"]]

,bus,carrier,p_set
name,,,
ZA0,ZA0,AC,0.000000
ZA0 industry electricity,ZA0,industry electricity,2.619179


In [13]:
n_electricity = n.copy()
n_electricity.optimize(solver_name=SOLVER, threads=1, log_to_console=False);
# Solar supplies the daytime load; OCGT fills the remaining gaps.

### Plot directly from statistics

Use `n.statistics.<metric>.iplot()` for the default interactive chart, or choose
`.iplot.area()`, `.iplot.line()` or `.iplot.bar()`. Filter on `bus_carrier` to
keep electricity and hydrogen units separate. `color="carrier"` keeps their
individual contributions visible. The Carrier colours defined above are reused.
See the [PyPSA statistics plotting guide](https://docs.pypsa.org/v1.2.4/user-guide/plotting/charts/).

This area chart shows electricity supply over time. The fixed Generator
capacities are listed in the component table above; no investment is chosen yet.

In [14]:
# Supply keeps injections positive; the battery balance below also shows withdrawals.
fig = n_electricity.statistics.supply.iplot.area(
    bus_carrier="AC", color="carrier", height=350)
fig.update_yaxes(title_text="Electricity supply [MW_el]")
fig.show()

## 2. Add a battery as a StorageUnit

Follow [attach_storageunits](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/add_extra_components.py#L228-L263)
and [the storage configuration](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/config.default.yaml#L197-L217).
The workflow supports this representation through `extendable_carriers.StorageUnit: [battery]`.
Here the first dispatch comparison fixes its power to 4 MW. With `max_hours: 6`,
its energy capacity is 24 MWh. Battery-inverter efficiency is a round-trip value;
the code applies its square root to charging and discharging, as in the source.

In [15]:
# Split the round-trip efficiency equally between charging and discharging.
inverter_efficiency = elec_costs.at["battery inverter", "efficiency"] ** 0.5
n.add("StorageUnit", nodes + " battery",
      bus=nodes,
      carrier="battery",
      p_nom=4.0,
      max_hours=config["electricity"]["max_hours"]["battery"],
      efficiency_store=inverter_efficiency,
      efficiency_dispatch=inverter_efficiency,
      cyclic_state_of_charge=True,
      capital_cost=elec_costs.at["battery", "capital_cost"],
      marginal_cost=elec_costs.at["battery", "marginal_cost"])
n_battery = n.copy()
n_battery.optimize(solver_name=SOLVER, threads=1, log_to_console=False);
# Expect daytime charging and discharge in higher residual-load hours.

In [16]:
fig = n_battery.storage_units_t.state_of_charge.plot(
    height=300, labels={"value": "Battery inventory [MWh]"},
    color_discrete_sequence=[n.carriers.at["battery", "color"]])
fig.show()

In [17]:
# Positive values inject electricity; negative values include charging and Loads.
fig = n_battery.statistics.energy_balance.iplot.area(bus_carrier="AC", color="carrier", height=350)
fig.update_yaxes(title_text="Power balance [MW_el]")
fig.show()

## 3. Add electrolysis and constant hydrogen export

The `H2 Electrolysis` Link connects the electricity bus to the `<node> H2` bus, following
[add_hydrogen](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/prepare_sector_network.py#L403-L423) and
[electrolysis Link construction](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/prepare_sector_network.py#L650-L683).

Hydrogen demand comes from the existing **exogenous export** option:
[export bus and port Links](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/add_export.py#L85-L103),
[export Load](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/add_export.py#L149-L157) and
[constant profile: TWh × 1e6 / 8760](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/add_export.py#L168-L174).
`export.h2export: 0.00876` means 1 MW_H2 at every hour, or 24 MWh over this day.
It is an annual-rate parameter; the 24-hour notebook does not deliver the entire annual volume.

In [18]:
n.add("Bus", nodes + " H2", location=nodes, carrier="H2", x=25.0, y=-29.0)

n.add("Link", nodes + " H2 Electrolysis",
      bus0=nodes,
      bus1=nodes + " H2",
      carrier="H2 Electrolysis",
      p_nom=6.0,
      efficiency=costs.at["electrolysis", "efficiency"],
      capital_cost=costs.at["electrolysis", "fixed"])

n.add("Bus", "H2 export bus", carrier="H2 export", location="Earth")

n.add("Link", nodes + " H2 export",
      bus0=nodes + " H2",
      bus1="H2 export bus",
      carrier="H2",
      p_nom=10.0)
# TWh/year × 1e6 MWh/TWh / 8,760 h gives the constant MW_H2 delivery.
export_profile = pd.Series(config["export"]["h2export"] * 1e6 / 8760, index=n.snapshots)

n.add("Load", "H2 export load", bus="H2 export bus", carrier="H2 export", p_set=export_profile)
n_h2_flat = n.copy()
n_h2_flat.optimize(solver_name=SOLVER, threads=1, log_to_console=False);
# With no H2 inventory, electrolysis follows the constant export rate.
# At efficiency 0.7, 1 MW_H2 needs about 1.43 MW_el, including at night.

## 4. Add the hydrogen tank

The naming and cyclic condition follow
[H2 Store Tank](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/prepare_sector_network.py#L789-L802).
Unlike the battery StorageUnit, its energy capacity is independent of the conversion power.
Predict how the tank changes electrolysis and battery use before running the next cells.

In [19]:
n.add("Store", nodes + " H2 Store Tank",
      bus=nodes + " H2",
      carrier="H2 Store Tank",
      e_nom=12.0,
      e_cyclic=True,
      capital_cost=costs.at["hydrogen storage tank type 1 including compressor", "fixed"])

n_h2_store = n.copy()

n_h2_store.optimize(solver_name=SOLVER, threads=1, log_to_console=False);
# Expect more electrolysis when renewables are available. Export remains constant.
# The battery can shift electricity, while the tank shifts hydrogen delivery.

In [20]:
# Link p0 is electricity input; StorageUnit p is positive when discharging.
dispatch = pd.DataFrame({
    "electrolysis without tank [MW_el]": n_h2_flat.links_t.p0["ZA0 H2 Electrolysis"],
    "electrolysis with tank [MW_el]": n_h2_store.links_t.p0["ZA0 H2 Electrolysis"],
    "battery discharge [MW_el]": n_h2_store.storage_units_t.p["ZA0 battery"],
})
fig = dispatch.plot(
    height=300,
    labels={"value": "Power [MW_el]"},
    line_dash="variable",
    line_dash_sequence=["dash", "solid", "solid"],
    color_discrete_sequence=[n.carriers.at["H2 Electrolysis", "color"],
                             n.carriers.at["H2 Electrolysis", "color"],
                             n.carriers.at["battery", "color"]])
fig.show()

In [21]:
stored = pd.DataFrame({
    "battery [MWh_el]": n_h2_store.storage_units_t.state_of_charge["ZA0 battery"],
    "hydrogen tank [MWh_H2]": n_h2_store.stores_t.e["ZA0 H2 Store Tank"],
})
fig = stored.plot(
    height=300,
    labels={"value": "Energy stored [carrier-labelled MWh]"},
    color_discrete_sequence=n.carriers.loc[["battery", "H2 Store Tank"], "color"].to_list())
fig.show()

In [26]:
cases = {"electricity": n_electricity, "battery": n_battery,
         "H2 without tank": n_h2_flat, "battery and H2 tank": n_h2_store}

summary = pd.DataFrame(index=cases)

for name, network in cases.items():
    weights = network.snapshot_weightings.generators
    summary.loc[name, "OCGT electricity [MWh]"] = network.generators_t.p["ZA0 OCGT"].mul(weights).sum()
    summary.loc[name, "operating cost [EUR/day]"] = network.statistics.opex(nice_names=False).sum()
# Compare the first two rows at the same electricity service and the last two at the same H2 service.
# Adding hydrogen changes demand, so its cost difference is not a pure flexibility saving.
summary.round(2)

,OCGT electricity [MWh],operating cost [EUR/day]
electricity,53.56,6186.42
battery,30.79,3558.95
H2 without tank,52.83,6104.43
battery and H2 tank,42.19,4874.83


In [27]:
# Compare equal H2 service; system_cost includes fixed-asset costs as well as operation.
h2_cases = pypsa.NetworkCollection({
    "without H2 tank": n_h2_flat,
    "with H2 tank": n_h2_store,
})

fig = h2_cases.statistics.system_cost.iplot.bar(
    x="network", y="value", color="carrier", height=350,
    labels={"network": "Hydrogen case"})
fig.show()

In [28]:
# Snapshot weights integrate power to MWh over this modelled day; H2 is separate below.
n_h2_store.statistics.energy_balance(bus_carrier="AC", groupby=["carrier"], nice_names=False)

component    carrier             
Generator    OCGT                    42.19220
             onwind                  44.80000
             solar                   91.14905
Link         H2 Electrolysis        -34.28571
Load         AC                     -79.18792
             industry electricity   -62.86030
StorageUnit  battery                 -1.80731
dtype: float64

In [29]:
n_h2_store.statistics.energy_balance(bus_carrier="H2", groupby=["carrier"], nice_names=False)

component  carrier        
Link       H2                -24.0
           H2 Electrolysis    24.0
dtype: float64

## 5. Add water as a second electrolysis input

Use the workshop assumption **20 L of water per kg_H2**. PyPSA-Earth's
[hydrogen LHV convention](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/config.default.yaml#L690-L693) is 33.3333 MWh/t_H2.
That is about 0.0333333 MWh/kg, so the water requirement is about **0.6 m³/MWh_H2**.
The Link is normalised to its electricity input: at efficiency 0.7,
`efficiency2 = -0.6 × 0.7 ≈ -0.42 m³/MWh_el`.

The negative sign consumes water at `bus2` while positive `efficiency` produces hydrogen
at `bus1`. This follows PyPSA-Earth's
[negative efficiency2 for an additional input](https://github.com/pypsa-meets-earth/pypsa-earth/blob/d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f/scripts/prepare_sector_network.py#L459-L470)
pattern. **The explicit water bus is a teaching extension of the Link pattern**;
this revision's standard `add_hydrogen` does not attach one.

The water bus uses volumetric units: supply is m³/h and its integrated balance is m³.
Those water quantities must not be labelled MW/MWh. For a transparent initial case,
water supply is ample at an illustrative 1 EUR/m³.

In [31]:
# Convert litres/kg_H2 to m3/MWh_H2, then normalise to the Link's bus0 input.
water_l_per_kg_h2 = 20.0
h2_mwh_per_kg = 33.3333 / 1000
water_m3_per_mwh_h2 = water_l_per_kg_h2 / 1000 / h2_mwh_per_kg
electrolysis_efficiency = costs.at["electrolysis", "efficiency"]
water_input_per_mwh_el = water_m3_per_mwh_h2 * electrolysis_efficiency
pd.Series({"water [m3/MWh_H2]": water_m3_per_mwh_h2,
           "efficiency2 [m3/MWh_el]": -water_input_per_mwh_el})

,0
water [m3/MWh_H2],0.600001
efficiency2 [m3/MWh_el],-0.420000


In [32]:
# Copy the unsolved tank-case builder, preserving the same capacities and H2 delivery.
n_water = n.copy()
n_water.add("Bus", nodes + " water", location=nodes, carrier="water", unit="m3")
# For this carrier, p_nom is a water rate in m3/h and marginal_cost is EUR/m3.
n_water.add("Generator", nodes + " water supply",
      bus=nodes + " water",
      carrier="water supply",
      p_nom=10.0,
      marginal_cost=1.0)
n_water.add("Link", nodes + " H2 Electrolysis",
      overwrite=True,
      bus0=nodes,
      bus1=nodes + " H2",
      bus2=nodes + " water",
      carrier="H2 Electrolysis",
      p_nom=6.0,
      efficiency=electrolysis_efficiency,
      efficiency2=-water_input_per_mwh_el,
      capital_cost=costs.at["electrolysis", "fixed"])
n_water.optimize(solver_name=SOLVER, threads=1, log_to_console=False);
cases["water input"] = n_water
# Ample water preserves the energy service and adds water-supply cost.

In [33]:
weights = n_water.snapshot_weightings.generators
h2_produced_mwh = (-n_water.links_t.p1["ZA0 H2 Electrolysis"]).mul(weights).sum()
water_used_m3 = n_water.links_t.p2["ZA0 H2 Electrolysis"].mul(weights).sum()
h2_produced_kg = h2_produced_mwh / h2_mwh_per_kg
# Expect about 720 kg_H2, 14.4 m3 of water, and 20 L/kg_H2 over the day.
# p2 is positive for withdrawal from the water bus; efficiency2 is negative.
pd.Series({"H2 production [kg]": h2_produced_kg,
           "water input [m3]": water_used_m3,
           "water input [L/kg_H2]": water_used_m3 * 1000 / h2_produced_kg})

,0
H2 production [kg],720.000720
water input [m3],14.400014
water input [L/kg_H2],20.000000


In [34]:
fig = n_water.links_t.p2[["ZA0 H2 Electrolysis"]].rename(
    columns={"ZA0 H2 Electrolysis": "water input [m3/h]"}).plot(
    height=300, labels={"value": "Water input [m3/h]"},
    color_discrete_sequence=[n.carriers.at["water supply", "color"]])
fig.show()

The next table isolates the cost of the explicit water input at the same
electricity and hydrogen service and with the same fixed tank. Ample water adds
its supply cost without changing the feasible electricity/H2 operating choices.

**Discuss:** Which additional assumptions would matter if water were scarce or
required electricity for pumping, treatment or desalination?

In [35]:
water_cost_impact = pd.Series({
    "water supply [EUR/day]": n_water.statistics.opex(
        components=["Generator"], carrier="water supply", nice_names=False).sum(),
    "system cost change [EUR/day]": (
        n_water.statistics.system_cost(nice_names=False).sum()
        - n_h2_store.statistics.system_cost(nice_names=False).sum()),
})
water_cost_impact.round(2)

,0
water supply [EUR/day],14.4
system cost change [EUR/day],14.4


**Try next:** reduce the water-supply `p_nom` from 10 to 1 m³/h and rerun this case.
Rerun the water-case construction and analysis cells after changing that value.
This limits the electrolysis rate, although the daily water volume can still cover demand.
Predict how the hydrogen tank and electricity battery respond. The 20 L/kg is the water
input specified for this exercise; no water recovery or fuel-cell recycling is modelled.

## Save your network for C

Save this version under `work/networks/`. The bundled starter stays available as a reference.
In C, select `starter_source = "work"` to continue from your version, or `"bundled"`
to use the course starter.

### Interpretation and recap

Discuss these questions using the tables and plots above. Use them for the joint A–C recap and quiz after the break.

1. Which inventory supplies electricity directly, and which shifts the timing of electrolysis? How do `StorageUnit` and `Store` represent power and energy limits?
2. Compare the tank and no-tank cases at the same H2 delivery. Why can the battery curve change too? Trace the electricity input, H2 output and water-input units.
3. Why do electricity, hydrogen and water at the same location need separate balances? Which conversions and rate limits must be represented explicitly?

**Later reflection:** Why is one synthetic day insufficient to study seasonal storage? Return to this question in the final model critique, after the country comparison.


In [36]:
n_water.meta = {"tutorial": "Synthetic 24-hour starter; not a full PyPSA-Earth run",
          "source_commit": "d9c6deb81ed9a03a166ac7acdc15532d8c9ecd4f",
          "water_l_per_kg_h2": water_l_per_kg_h2, "h2_mwh_per_kg": h2_mwh_per_kg}
(work_dir / "networks").mkdir(parents=True, exist_ok=True)
n_water.export_to_netcdf(work_dir / "networks/elec_s_1_ec_lcopt_1h_sector.nc");